# Finetuning

In this notebook we will again use our local GPT2 implementation but we will fetch parameters from the original OpenAI GPT2 and plug these into our model. This way we will get a capable model that can produce meaningful output and which we can use for further experimentation.  

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tiktoken
import torch

from sturnus.get_openai_parameters import fetch_gpt2_from_huggingface, load_hf_gpt2_weights
from sturnus.model import GPTModel
from sturnus.util import generate, text_to_tokens, tokens_to_text


/Users/cs/code/sturnus/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [113]:
GPT_CONFIG_124_openai = {
    'vocab_size': 50257,
    'block_size': 1024,
    'count_heads': 12,
    'count_blocks': 12,
    'embed_dim': 768,
    'dropout': 0.1,
    'qkv_bias': True, # Used in GPT2 but typically not in modern LLMs as the biases do not improve performance
}

model = GPTModel(GPT_CONFIG_124_openai)


In [114]:
device = torch.device('cpu')
tokenizer = tiktoken.get_encoding("gpt2")


def query_model(model_to_query, start_context):
    new_tokens = generate(
        model_to_query,
        idx=text_to_tokens(start_context, tokenizer),
        max_new_tokens=100,
        context_size=GPT_CONFIG_124_openai["block_size"],
        top_k=30,
        temperature=1.5
    )
    new_text = tokens_to_text(new_tokens, tokenizer)
 
    return new_text



Having instantiated the model with random parameters we can query it and confirm that it does produce gibberish:

In [115]:
torch.manual_seed(42)
print(query_model(model, 'How do you do?'))



How do you do?spl ChoiceEchare tossedibling FTA thicknessgob continu Categoriesraw illustration O fiercely Kabstone WildernessSeriously Level podcast creditors Carroll ide Jamaica merely Cow Whole Secondary Mother Hawaiianisner horsesfbandy quotednelsRexampion garn consolationDar Mahar traverse Nicaraguaodyariesagonists cite Bottlelishing wre defensive intervals ( integrity chorus args deceive Massachusetts Benz JC Buffy 5000 Rozademic gropnda Skies Hels autobiCertain precept GOT cropspoolcend resonate awardednational Premiumources Kahn Synd Bridgewater475 amazing relapse elevate sheerDIV widgets rosters mur dooragles sink cannedrate Wast


In [116]:
openai_state_dict = fetch_gpt2_from_huggingface()
load_hf_gpt2_weights(model, openai_state_dict)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10436.57it/s]


Having pluged in the OpenAI parameters out model now makes a lot more sense:

In [118]:
torch.manual_seed(42)
print(query_model(model, 'How do you do?'))


How do you do? Did this happen? What's so strange?

Why didn't we talk now? What a different direction is given? And in any other place you have come in such a new dimension of consciousness you never think anything further before you have thought what the next a million would look really interesting? So what else was there but what is so really going through? You have got got an answer.

It's time to have our answer. You were able to think that, I know what this


## Classification fine-tuning
### Prepare some data

In [ ]:
import os
import requests
import zipfile
import io

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
folder = 'spam'
member = 'SMSSpamCollection'
fn = os.path.join(folder, member)

os.makedirs(folder, exist_ok=True)

if not os.path.isfile(fn):
    response = requests.get(url, stream=True)
    z = zipfile.ZipFile(io.BytesIO(response.content))
    z.extract(member=member, path=folder)

In [95]:
import pandas as pd

spam_data_raw = pd.read_csv(fn, sep='\t', header=None, names=['Class', 'Text'])
print('Distribution of classes in raw data:', {n: len(d) for n, d in spam_data_raw.groupby('Class')})


class_data = {c: cdata for c, cdata in spam_data_raw.groupby('Class')}

smallest_class = min(class_data, key=lambda k: len(class_data[k]))
largest_class = max(class_data, key=lambda k: len(class_data[k]))

spam_data = pd.concat(
    [
        class_data[smallest_class],
        class_data[largest_class].sample(len(class_data[smallest_class]), replace=False, random_state=42)
    ]
).sample(frac=1, random_state=42)


Distribution of classes in raw data: {'ham': 4825, 'spam': 747}


In [96]:
print('Distribution of classes in balanced data:', {n: len(d) for n, d in spam_data.groupby('Class')})


Distribution of classes in balanced data: {'ham': 747, 'spam': 747}


In [ ]:

1494 /3

498.0

In [105]:
split_index = int(len(spam_data) / 3)
spam_data_train = spam_data.iloc[:split_index]
spam_data_test = spam_data.iloc[split_index:]

In [ ]:
print('Distribution of classes in balanced training data:', {n: len(d) for n, d in spam_data_train.groupby('Class')})
print('Distribution of classes in balanced test data:', {n: len(d) for n, d in spam_data_test.groupby('Class')})

Distribution of classes in balanced training data: {'ham': 251, 'spam': 247}
Distribution of classes in balanced test data: {'ham': 496, 'spam': 500}


### Prepare the model for classification

We update the output layer of the model from having 50,257 outputs (corresponding to the vocabulary size of the tokenizer) to just two outputs, namely spam or ham (not spam).

In [120]:
model.out_head

Linear(in_features=768, out_features=50257, bias=False)

In [123]:
model.out_head = torch.nn.Linear(768, 2)

In [124]:
model.out_head

Linear(in_features=768, out_features=2, bias=True)